# Build Three Math Games

For this challenge, you need to create three math games using Python that do the following:

1) Scatter plot game:
- Randomly generate points on a graph and the player has to input the (x,y) coordinates
- For added difficulty, make the graph larger

2) Algebra practice game:
- Generate one-step and two-step problems with random integer values and the player has to input the answer
- Use positive and negative values. For added difficulty, make the numbers larger

3) Projectile game:
- Display a "wall" with random height and location.
- Player has to move sliders to adjust a parabolic path to clear the wall
- For added difficulty, make a second level where players enter a, b, and c without sliders

# Scatter Plot Game

- Randomly generate points on a graph and the player has to input the (x,y) coordinates
- For added difficulty, make the graph larger

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import random
import ipywidgets as widgets
from IPython.display import display, clear_output

def scatter_game():
    clear_output()

    # Generate random points
    num_points = 5
    x_vals = sorted([random.randint(-20, 20) for _ in range(num_points)])
    y_vals = [random.randint(-20, 20) for _ in range(num_points)]
    points = list(zip(x_vals, y_vals))

    # Show the plot
    plt.figure(figsize=(8, 8))
    plt.scatter(x_vals, y_vals, color='red')
    plt.axhline(0, color='gray')
    plt.axvline(0, color='gray')
    plt.grid(True)
    plt.xlim(-25, 25)
    plt.ylim(-25, 25)
    plt.title("Scatter Plot Game: Guess the Coordinates (Left to Right)")
    plt.show()

     # Input fields for guesses
    guess_widgets = []
    for i in range(num_points):
        x_input = widgets.IntText(description=f'X{i+1}:')
        y_input = widgets.IntText(description=f'Y{i+1}:')
        guess_widgets.append((x_input, y_input))

    submit_button = widgets.Button(description="Submit Guesses")
    reset_button = widgets.Button(description="Reset Game", button_style='warning')
    output = widgets.Output()

    def check_guesses(b):
        correct = 0
        with output:
            clear_output()
            print("Results:")
            for i, (guess_x, guess_y) in enumerate(guess_widgets):
                user_x = guess_x.value
                user_y = guess_y.value
                actual_x, actual_y = points[i]
                close = abs(user_x - actual_x) <= 1 and abs(user_y - actual_y) <= 1
                result = "✔ Correct" if close else f"✘ Wrong (Actual: {actual_x}, {actual_y})"
                print(f"Point {i+1}: Your guess ({user_x}, {user_y}) — {result}")
                if close:
                    correct += 1
            print(f"\nYou got {correct} out of {num_points} correct!")

    def reset_game(b):
        scatter_game()

    submit_button.on_click(check_guesses)
    reset_button.on_click(reset_game)

    # Display the inputs
    inputs_layout = []
    for x_input, y_input in guess_widgets:
        inputs_layout.append(widgets.HBox([x_input, y_input]))

    display(widgets.VBox(inputs_layout + [widgets.HBox([submit_button, reset_button]), output]))

# Run the game
scatter_game()

# Algebra Practice Game

- Generate one-step and two-step problems with random integer values and the player has to input the answer
- Use positive and negative values. For added difficulty, make the numbers larger

In [ ]:
import random
from fractions import Fraction
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets
output = widgets.Output()
question_label = widgets.Label()
answer_input = widgets.Text(placeholder="Type your answer (e.g. -2 or 1/2)")
submit_button = widgets.Button(description="Submit", button_style='success')
reset_button = widgets.Button(description="Reset Game", button_style='warning')
score_label = widgets.Label()
feedback_label = widgets.Label()

# Game state
score = 0
total_questions = 0
current_answer = None

def generate_question():
    global current_answer
    q_type = random.choice(["one_step", "two_step"])
    if q_type == "one_step":
        x = random.randint(-20, 20)
        a = random.randint(-10, 10)
        while a == 0:
            a = random.randint(-10, 10)
        b = a * x
        question_label.value = f"Solve: {a}x = {b}"
        current_answer = x
    else:  # two-step
        x = random.randint(-20, 20)
        a = random.randint(-10, 10)
        b = random.randint(-20, 20)
        while a == 0:
            a = random.randint(-10, 10)
        c = a * x + b
        question_label.value = f"Solve: {a}x + {b} = {c}"
        current_answer = x

def check_answer(_):
    global score, total_questions
    user_input = answer_input.value.strip()

    try:
        # Safely parse input as fraction or float
        if '/' in user_input:
            user_value = float(Fraction(user_input))
        else:
            user_value = float(user_input)

        if abs(user_value - current_answer) < 1e-6:
            score += 1
            feedback_label.value = "Correct!"
        else:
            feedback_label.value = f"Incorrect. The correct answer was {current_answer}"
        total_questions += 1
        score_label.value = f"Score: {score}/{total_questions}"
    except:
        feedback_label.value = "Invalid input. Please enter a number or fraction like 1/2."

    answer_input.value = ""
    generate_question()

def reset_game(_):
    global score, total_questions
    score = 0
    total_questions = 0
    score_label.value = "Score: 0/0"
    feedback_label.value = ""
    answer_input.value = ""
    generate_question()

# Bind buttons
submit_button.on_click(check_answer)
reset_button.on_click(reset_game)

# Initial state
generate_question()

# Display UI
with output:
    display(question_label, answer_input, widgets.HBox([submit_button, reset_button]), feedback_label, score_label)

display(output)



# Projectile Game

- Display a "wall" with random height and location.
Player has to move sliders to adjust a parabolic path to clear the wall
- For added difficulty, make a second level where players enter a, b, and c without sliders

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider
import math
import random

# Random wall
wall_x = random.randint(5, 15)
wall_height = random.randint(10, 30)

# Shared logic
def run_projectile_game(a, b, c):
    discriminant = b**2 - 4 * a * c
    if discriminant < 0:
        time_to_ground = 0
    else:
        t1 = (-b + math.sqrt(discriminant)) / (2 * a)
        t2 = (-b - math.sqrt(discriminant)) / (2 * a)
        time_to_ground = max(t1, t2)

    x = np.linspace(0, max(wall_x + 5, time_to_ground + 1), 500)
    y = a * x**2 + b * x + c
    y_at_wall = a * wall_x**2 + b * wall_x + c

    if a > 0:
        outcome = "What kind of rocket is this?! That thing's going to the moon!"
    elif wall_x > time_to_ground:
        outcome = "Oops! The rocket lands before reaching the wall."
    elif y_at_wall > wall_height:
        outcome = "Success! The rocket clears the wall!"
    else:
        outcome = "Ouch! The rocket crashed into the wall."

    # Plotting
    plt.figure(figsize=(10, 5))
    plt.plot(x, y, label="Rocket Path", color='black')
    plt.plot([wall_x, wall_x], [0, wall_height], color='red', linewidth=3, label="Wall")
    plt.axhline(y=0, color='blue')
    plt.ylim(bottom=0)
    plt.title(outcome)
    plt.xlabel("Distance")
    plt.ylabel("Height")
    plt.legend()
    plt.grid(True)
    plt.show()

# Interactive Mode
def projectile_game_interactive(b, c):
    a = -4.9
    run_projectile_game(a, b, c)

# Manual Mode
def projectile_game_manual():
    print("Manual Projectile Launch")
    try:
        a = float(input("Enter a (gravity, usually -4.9): "))
        b = float(input("Enter b (initial velocity): "))
        c = float(input("Enter c (initial height): "))
        run_projectile_game(a, b, c)
    except ValueError:
        print("Invalid input. Please enter numeric values.")

# Main menu
def game_menu():
    print("=== Projectile Game ===")
    print("1. Interactive Mode (with sliders)")
    print("2. Manual Mode (type values)")
    choice = input("Choose a mode (1 or 2): ")

    if choice == "1":
        interactive_plot = interactive(
            projectile_game_interactive,
            b=FloatSlider(value=20, min=5, max=50, step=1, description='Velocity (b)'),
            c=FloatSlider(value=0, min=0, max=20, step=1, description='Height (c)')
        )
        display(interactive_plot)
    elif choice == "2":
        projectile_game_manual()
    else:
        print("Invalid choice. Please select 1 or 2.")


game_menu()


